# Notebook 02: Baseline & Statistical Forecasting Models
**Models Covered**:
1. **Naïve Persistence Forecast** (`y_hat(t) = y(t-1)`)
2. **Moving Average Forecast** (`7-Day Rolling Mean`)
3. **Exponential Smoothing (Holt-Winters Damped Trend & Seasonality)**
4. **ARIMA / SARIMA (Seasonal Autoregressive Integrated Moving Average with Exogenous Net Flow)**

### Evaluation Protocol
- Strict time-based split (final `180` continuous days held out as out-of-sample test window).
- Walk-forward rolling refit to handle the 2024→2025 policy regime transition.
- Evaluated on `MAE`, `RMSE`, `MAPE (%)`, `Forecast Accuracy (%)`, `Horizon Error (1d, 7d, 14d, 30d)`, `Forecast Stability Index`, and `Model Robustness`.

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Optional statsmodels import for full parametric SARIMAX & Holt-Winters fitting
try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

from app import load_and_prepare_data, engineer_features, run_walk_forward_backtest

df = load_and_prepare_data("../data/HHS_Unaccompanied_Alien_Children_Program.csv", "data/HHS_Unaccompanied_Alien_Children_Program.csv", "HHS_Unaccompanied_Alien_Children_Program.csv")
feat = engineer_features(df)

lb_care, pred_care, _ = run_walk_forward_backtest(feat, target="hhs_care", test_days=180)
lb_disc, pred_disc, _ = run_walk_forward_backtest(feat, target="hhs_discharged", test_days=180)

stat_models = ["ARIMA / SARIMA", "Exponential Smoothing", "Naive Persistence", "Moving Average (7d)"]
print("=== BASELINE & STATISTICAL MODELS: CHILDREN IN HHS CARE (180d Walk-Forward) ===")
print(lb_care.loc[stat_models, ["Family", "MAE", "RMSE", "MAPE (%)", "Forecast Accuracy (%)", "Horizon 7d MAE", "Horizon 14d MAE", "Horizon 30d MAE", "Forecast Stability Index", "Model Robustness"]].to_string())

=== BASELINE & STATISTICAL MODELS: CHILDREN IN HHS CARE (180d Walk-Forward) ===
                            Family    MAE   RMSE  MAPE (%)  Forecast Accuracy (%)  Horizon 7d MAE  Horizon 14d MAE  Horizon 30d MAE  Forecast Stability Index  Model Robustness
Model                                                                                                                                                                              
ARIMA / SARIMA         Statistical   4.82   6.21      0.22                  99.78            6.59             7.43             8.71                      95.5              99.8
Exponential Smoothing  Statistical   5.51   7.12      0.25                  99.75            7.53             8.49             9.96                     100.0              99.8
Naive Persistence         Baseline   6.93   8.56      0.31                  99.69           10.66            12.43            15.14                      99.8              99.6
Moving Average (7d)       Baseline  

In [2]:
print("=== BASELINE & STATISTICAL MODELS: DISCHARGE DEMAND (180d Walk-Forward) ===")
print(lb_disc.loc[stat_models, ["Family", "MAE", "RMSE", "MAPE (%)", "Forecast Accuracy (%)", "Horizon 7d MAE", "Horizon 14d MAE", "Horizon 30d MAE", "Forecast Stability Index", "Model Robustness"]].to_string())

# Plot Walk-Forward Predictions vs Actuals
plt.figure(figsize=(13, 5))
plt.plot(pred_care.index, pred_care["Actual"], label="Actual HHS Care Load", color="#0f172a", linewidth=2.5)
plt.plot(pred_care.index, pred_care["ARIMA / SARIMA"], label="ARIMA / SARIMA (MAE: 4.82)", color="#2563eb", linestyle="--")
plt.plot(pred_care.index, pred_care["Exponential Smoothing"], label="Exponential Smoothing (MAE: 5.51)", color="#0d9488", linestyle=":")
plt.plot(pred_care.index, pred_care["Moving Average (7d)"], label="Moving Average 7d (MAE: 23.23)", color="#dc2626", alpha=0.7)
plt.title("180-Day Walk-Forward Backtest: Statistical vs. Baseline Models (Children in HHS Care)")
plt.ylabel("Children in HHS Care")
plt.legend()
plt.tight_layout()
plt.show()

=== BASELINE & STATISTICAL MODELS: DISCHARGE DEMAND (180d Walk-Forward) ===
                            Family   MAE  RMSE  MAPE (%)  Forecast Accuracy (%)  Horizon 7d MAE  Horizon 14d MAE  Horizon 30d MAE  Forecast Stability Index  Model Robustness
Model                                                                                                                                                                            
ARIMA / SARIMA         Statistical  3.60  4.83     28.62                  71.38            4.92             5.55             6.51                      90.5              81.6
Naive Persistence         Baseline  3.96  5.25     31.43                  68.57            6.09             7.10             8.65                      99.9              74.9
Moving Average (7d)       Baseline  4.05  5.32     31.51                  68.49            6.23             7.26             8.85                      85.0              80.2
Exponential Smoothing  Statistical  4.10  5.39    